# 05 - Savings Recommendations

Train a Decision Tree classifier + rule engine to generate personalized savings tips.

**Approach**: Two-part system:
1. **Decision Tree**: Classify spending profiles into risk levels (low/medium/high) based on spending features
2. **Rule Engine**: Generate specific actionable recommendations based on category spending patterns

**Pipeline**:
1. Feature engineering from spending data
2. Train Decision Tree on synthetic finance dataset
3. Build rule-based recommendation templates
4. Export to TFLite + recommendation templates JSON for Flutter

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

from preprocessing import APP_CATEGORIES
from export_tflite import export_keras_to_tflite, verify_tflite_model

print(f'TensorFlow version: {tf.__version__}')

## 1. Load & Prepare Data

In [ ]:
# Load savings training data (preprocessed in notebook 01)
df = pd.read_csv('../data/processed/savings_train.csv')
print(f'Dataset: {len(df)} rows')
print(f'\nColumns: {list(df.columns)}')
print(f'\nSample:')
df.head()

In [ ]:
# Feature engineering for savings risk classification
# We compute features that Flutter can also compute from transaction history

def engineer_savings_features(df):
    features = pd.DataFrame()
    
    # Income-to-expense ratio
    if 'Monthly_Income' in df.columns and 'Monthly_Expenses' in df.columns:
        features['expense_ratio'] = df['Monthly_Expenses'] / df['Monthly_Income'].clip(lower=1)
    elif 'income' in df.columns and 'expenses' in df.columns:
        features['expense_ratio'] = df['expenses'] / df['income'].clip(lower=1)
    
    # Savings rate
    if 'Savings' in df.columns and 'Monthly_Income' in df.columns:
        features['savings_rate'] = df['Savings'] / df['Monthly_Income'].clip(lower=1)
    elif 'savings' in df.columns and 'income' in df.columns:
        features['savings_rate'] = df['savings'] / df['income'].clip(lower=1)
    else:
        features['savings_rate'] = 1.0 - features['expense_ratio']
    
    # Discretionary spending ratio (dining + entertainment + shopping)
    disc_cols = [c for c in df.columns if any(k in c.lower() for k in ['dining', 'entertainment', 'shopping', 'recreation', 'leisure'])]
    if disc_cols:
        income_col = 'Monthly_Income' if 'Monthly_Income' in df.columns else 'income'
        features['discretionary_ratio'] = df[disc_cols].sum(axis=1) / df[income_col].clip(lower=1)
    else:
        features['discretionary_ratio'] = features['expense_ratio'] * 0.4  # estimate
    
    # Essential spending ratio (rent + utilities + groceries + transport)
    ess_cols = [c for c in df.columns if any(k in c.lower() for k in ['rent', 'utilities', 'groceries', 'transport', 'housing', 'mortgage'])]
    if ess_cols:
        income_col = 'Monthly_Income' if 'Monthly_Income' in df.columns else 'income'
        features['essential_ratio'] = df[ess_cols].sum(axis=1) / df[income_col].clip(lower=1)
    else:
        features['essential_ratio'] = features['expense_ratio'] * 0.6  # estimate
    
    # Credit utilization (if available)
    if 'Credit_Utilization_Ratio' in df.columns:
        features['credit_util'] = df['Credit_Utilization_Ratio']
    else:
        features['credit_util'] = 0.3  # default
    
    return features

features = engineer_savings_features(df)
print(f'Features: {list(features.columns)}')
print(f'Shape: {features.shape}')
features.describe()

In [ ]:
# Create savings risk labels
# 0 = healthy (savings rate > 20%)
# 1 = moderate (savings rate 5-20%)
# 2 = at_risk (savings rate < 5% or expense ratio > 90%)

def label_risk(row):
    if row['savings_rate'] > 0.20:
        return 0  # healthy
    elif row['savings_rate'] > 0.05:
        return 1  # moderate
    else:
        return 2  # at_risk

labels = features.apply(label_risk, axis=1).values
RISK_LABELS = ['healthy', 'moderate', 'at_risk']

print('Risk distribution:')
for i, label in enumerate(RISK_LABELS):
    print(f'  {label}: {(labels == i).sum()} ({(labels == i).mean():.1%})')

## 2. Train Decision Tree (sklearn) & Convert to Keras

In [ ]:
# Train/test split
X = features.values.astype(np.float32)
y = labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {len(X_train)} samples')
print(f'Test:  {len(X_test)} samples')

In [ ]:
# Train sklearn Decision Tree (for interpretability)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_scaled, y_train)

dt_pred = dt.predict(X_test_scaled)
print('Decision Tree Results:')
print(f'Accuracy: {accuracy_score(y_test, dt_pred):.4f}')
print(classification_report(y_test, dt_pred, target_names=RISK_LABELS))

# Print tree rules (for documentation)
print('\nDecision Rules:')
print(export_text(dt, feature_names=list(features.columns), max_depth=3))

In [ ]:
# Build equivalent Keras model (Dense network that mimics decision tree)
# This lets us export to TFLite
INPUT_DIM = X_train_scaled.shape[1]
NUM_CLASSES = 3

keras_model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(INPUT_DIM,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])

keras_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history = keras_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=10, restore_best_weights=True, monitor='val_accuracy'
        ),
    ],
)

keras_pred = np.argmax(keras_model.predict(X_test_scaled), axis=1)
print('\nKeras Model Results:')
print(f'Accuracy: {accuracy_score(y_test, keras_pred):.4f}')
print(classification_report(y_test, keras_pred, target_names=RISK_LABELS))

## 3. Recommendation Templates

In [ ]:
import json

# Rule-based recommendation templates
# These are used in Flutter alongside the risk classification

recommendations = {
    "risk_levels": {
        "healthy": {
            "title": "Great Financial Health",
            "description": "You're saving well! Here are ways to optimize further.",
            "icon": "emoji_events"
        },
        "moderate": {
            "title": "Room for Improvement",
            "description": "Your spending is manageable, but you could save more.",
            "icon": "trending_up"
        },
        "at_risk": {
            "title": "Action Needed",
            "description": "Your expenses are close to or exceeding your income.",
            "icon": "warning"
        }
    },
    "rules": [
        {
            "id": "high_dining",
            "condition": "category_ratio.dining > 0.15",
            "title": "Reduce Dining Expenses",
            "description": "Dining makes up {ratio}% of your spending. Try meal prepping to cut costs by 30-50%.",
            "category": "dining",
            "threshold": 0.15,
            "priority": "high",
            "potential_savings": 0.3
        },
        {
            "id": "high_entertainment",
            "condition": "category_ratio.entertainment > 0.10",
            "title": "Optimize Entertainment Budget",
            "description": "Entertainment is {ratio}% of spending. Consider free alternatives or shared subscriptions.",
            "category": "entertainment",
            "threshold": 0.10,
            "priority": "medium",
            "potential_savings": 0.25
        },
        {
            "id": "high_shopping",
            "condition": "category_ratio.shopping > 0.20",
            "title": "Curb Shopping Spending",
            "description": "Shopping is {ratio}% of your expenses. Try a 24-hour rule before non-essential purchases.",
            "category": "shopping",
            "threshold": 0.20,
            "priority": "high",
            "potential_savings": 0.35
        },
        {
            "id": "high_transport",
            "condition": "category_ratio.transport > 0.15",
            "title": "Lower Transport Costs",
            "description": "Transport is {ratio}% of spending. Consider carpooling, public transit, or biking.",
            "category": "transport",
            "threshold": 0.15,
            "priority": "medium",
            "potential_savings": 0.20
        },
        {
            "id": "high_groceries",
            "condition": "category_ratio.groceries > 0.25",
            "title": "Optimize Grocery Budget",
            "description": "Groceries are {ratio}% of spending. Plan meals, buy in bulk, and use coupons.",
            "category": "groceries",
            "threshold": 0.25,
            "priority": "medium",
            "potential_savings": 0.15
        },
        {
            "id": "no_savings",
            "condition": "savings_rate < 0.05",
            "title": "Start an Emergency Fund",
            "description": "You're saving less than 5%. Aim to save at least 10% of income as a safety net.",
            "category": null,
            "threshold": 0.05,
            "priority": "critical",
            "potential_savings": null
        },
        {
            "id": "spending_increase",
            "condition": "month_over_month_increase > 0.20",
            "title": "Spending Increased Significantly",
            "description": "Your spending increased {increase}% from last month. Review recent purchases for non-essentials.",
            "category": null,
            "threshold": 0.20,
            "priority": "high",
            "potential_savings": null
        },
        {
            "id": "high_utilities",
            "condition": "category_ratio.utilities > 0.12",
            "title": "Reduce Utility Bills",
            "description": "Utilities are {ratio}% of spending. Check for better plans, reduce usage, or switch providers.",
            "category": "utilities",
            "threshold": 0.12,
            "priority": "low",
            "potential_savings": 0.10
        }
    ]
}

print(f'Recommendation rules: {len(recommendations["rules"])}')
for r in recommendations['rules']:
    print(f'  [{r["priority"]:>8s}] {r["id"]}: {r["title"]}')

## 4. Export to TFLite

In [ ]:
# Save Keras model
keras_model.save('../models/saved/savings_advisor_keras')
print('Keras model saved.')

# Export TFLite
tflite_meta = export_keras_to_tflite(
    keras_model,
    output_path='../models/tflite/savings_advisor.tflite',
    quantize=True,
)
print(f'\nTFLite model: {tflite_meta["size_kb"]:.1f} KB')

In [ ]:
# Export config + recommendation templates
savings_config = {
    'input_dim': INPUT_DIM,
    'num_classes': NUM_CLASSES,
    'risk_labels': RISK_LABELS,
    'feature_names': list(features.columns),
    'scaler': {
        'mean': scaler.mean_.tolist(),
        'std': scaler.scale_.tolist(),
    },
    'metrics': {
        'accuracy': float(accuracy_score(y_test, keras_pred)),
    },
}

with open('../models/tflite/savings_config.json', 'w') as f:
    json.dump(savings_config, f, indent=2)

with open('../models/tflite/recommendation_templates.json', 'w') as f:
    json.dump(recommendations, f, indent=2)

print('Savings config saved.')
print('Recommendation templates saved.')

In [ ]:
# Verify TFLite model
sample = X_test_scaled[:5].astype(np.float32)
tflite_output = verify_tflite_model('../models/tflite/savings_advisor.tflite', sample)

keras_output = keras_model.predict(sample, verbose=0)

print(f'\nKeras vs TFLite comparison:')
for i in range(5):
    keras_risk = RISK_LABELS[np.argmax(keras_output[i])]
    tflite_risk = RISK_LABELS[np.argmax(tflite_output[i])]
    match = 'MATCH' if keras_risk == tflite_risk else 'MISMATCH'
    print(f'  Sample {i}: Keras={keras_risk}, TFLite={tflite_risk} [{match}]')

## Results Summary

### Model Architecture
```
Input(5) → Dense(16, relu) → Dense(8, relu) → Dense(3, softmax)
```

### Risk Levels
| Level | Savings Rate | Description |
|-------|-------------|-------------|
| healthy | > 20% | Good financial habits |
| moderate | 5-20% | Room for improvement |
| at_risk | < 5% | Spending close to income |

### Files Exported for Flutter
| File | Description |
|------|-------------|
| `savings_advisor.tflite` | Risk classifier |
| `savings_config.json` | Scaler params, feature names |
| `recommendation_templates.json` | Rule-based tips |

### Next: Copy exports to Flutter
```bash
cp ../models/tflite/savings_advisor.tflite ../../app/assets/models/
cp ../models/tflite/savings_config.json ../../app/assets/models/
cp ../models/tflite/recommendation_templates.json ../../app/assets/models/
```